In [24]:
pip install streamlit

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 10.1 MB 3.3 MB/s eta 0:00:01
     |████████████████████████████████| 427 kB 3.8 MB/s eta 0:00:01
     |████████████████████████████████| 98 kB 4.8 MB/s eta 0:00:011
     |████████████████████████████████| 447 kB 2.9 MB/s eta 0:00:01
     |████████████████████████████████| 731 kB 3.5 MB/s eta 0:00:01
     |████████████████████████████████| 31.2 MB 3.3 MB/s eta 0:00:01
     |████████████████████████████████| 212 kB 2.7 MB/s eta 0:00:01
     |████████████████████████████████| 64 kB 3.4 MB/s eta 0:00:01
     |████████████████████████████████| 4.7 MB 1.8 MB/s eta 0:00:01
     |████████████████████████████████| 11.3 MB 2.1 MB/s eta 0:00:01
     |████████████████████████████████| 451 kB 3.2 MB/s eta 0:00:01
     |████████████████████████████████| 134 kB 3.3 MB/s eta 0:00:01
     |████████████████████████████████| 90 kB 3.2 MB/s eta 0:00:01
     |███████████████████████████████

In [10]:
pip install sklearn

Defaulting to user installation because normal site-packages is not writeable
    ERROR: Command errored out with exit status 1:
     command: /Library/Developer/CommandLineTools/usr/bin/python3 -c 'import io, os, sys, setuptools, tokenize; sys.argv[0] = '"'"'/private/var/folders/00/45cm1jfj0nq_87yby0dthxb40000gn/T/pip-install-j7sd4f3m/sklearn_12a40fea7cee4c94abe08ae9bf3416fa/setup.py'"'"'; __file__='"'"'/private/var/folders/00/45cm1jfj0nq_87yby0dthxb40000gn/T/pip-install-j7sd4f3m/sklearn_12a40fea7cee4c94abe08ae9bf3416fa/setup.py'"'"';f = getattr(tokenize, '"'"'open'"'"', open)(__file__) if os.path.exists(__file__) else io.StringIO('"'"'from setuptools import setup; setup()'"'"');code = f.read().replace('"'"'\r\n'"'"', '"'"'\n'"'"');f.close();exec(compile(code, __file__, '"'"'exec'"'"'))' egg_info --egg-base /private/var/folders/00/45cm1jfj0nq_87yby0dthxb40000gn/T/pip-pip-egg-info-t5gvcpvc
         cwd: /private/var/folders/00/45cm1jfj0nq_87yby0dthxb40000gn/T/pip-install-j7sd4f3m/sklea

In [5]:
pip install pandas

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 10.8 MB 1.2 MB/s eta 0:00:01
     |████████████████████████████████| 349 kB 1.6 MB/s eta 0:00:01
     |████████████████████████████████| 5.3 MB 1.0 MB/s eta 0:00:01
     |████████████████████████████████| 510 kB 869 kB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [22]:
import pandas as pd
import ast
import re
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

# Load the dataset (adjust the filename if yours is different)
df = pd.read_csv('../data/recipes.csv')

# Look at the first 3 rows
display(df.head(3))

# Check how many rows we have and if there are missing values
print(df.info())

,Unnamed: 0,Title,Ingredients,Instructions,Image_Name,Cleaned_Ingredients
0,0,Miso-Butter Roast Chicken With Acorn Squash Pa...,"['1 (3½–4-lb.) whole chicken', '2¾ tsp. kosher...","Pat chicken dry with paper towels, season all ...",miso-butter-roast-chicken-acorn-squash-panzanella,"['1 (3½–4-lb.) whole chicken', '2¾ tsp. kosher..."
1,1,Crispy Salt and Pepper Potatoes,"['2 large egg whites', '1 pound new potatoes (...",Preheat oven to 400°F and line a rimmed baking...,crispy-salt-and-pepper-potatoes-dan-kluger,"['2 large egg whites', '1 pound new potatoes (..."
2,2,Thanksgiving Mac and Cheese,"['1 cup evaporated milk', '1 cup whole milk', ...",Place a rack in middle of oven; preheat to 400...,thanksgiving-mac-and-cheese-erick-williams,"['1 cup evaporated milk', '1 cup whole milk', ..."


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13501 entries, 0 to 13500
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Unnamed: 0           13501 non-null  int64 
 1   Title                13496 non-null  object
 2   Ingredients          13501 non-null  object
 3   Instructions         13493 non-null  object
 4   Image_Name           13501 non-null  object
 5   Cleaned_Ingredients  13501 non-null  object
dtypes: int64(1), object(5)
memory usage: 633.0+ KB
None


In [23]:
# A list of words we want to delete from the ingredients
measurements_to_remove = [
    'a', 'an', 'the', 'and', 'or', 'with', 'of', 'in', 'into', 'for', 'to', 'some', 'as', 'at', 'about',
    'of', 'about', 'accompaniment', 'cup', 'cups', 'oz', 'ounce', 'ounces', 'tsp', 'teaspoon', 'teaspoons',
    'tbsp', 'tablespoon', 'tablespoons', 'pound', 'pounds', 'lb', 'lbs',
    'gram', 'grams', 'g', 'kg', 'ml', 'liter', 'liters', 'pint', 'pints',
    'quart', 'quarts', 'gallon', 'gallons', 'pinch', 'dash', 'piece', 'pieces',
    'chopped', 'diced', 'minced', 'sliced', 'peeled', 'fresh', 'large', 'small'
]

def clean_ingredients(ingredient_text):
    try:
        if isinstance(ingredient_text, str) and ingredient_text.startswith('['):
            ingredients_list = ast.literal_eval(ingredient_text)
        else:
            ingredients_list = str(ingredient_text).split(',')
    except:
        return ""

    cleaned_list = []
    for item in ingredients_list:
        item = item.lower()
        item = re.sub(r'[^a-z\s]', '', item)
        words = item.split()
        filtered_words = [word for word in words if word not in measurements_to_remove]
        
        # Combine multi-word ingredients with an underscore (e.g., "rice vinegar" -> "rice_vinegar")
        cleaned_item = "_".join(filtered_words).strip('_')
        
        if cleaned_item:
            cleaned_list.append(cleaned_item)
            
    return " ".join(cleaned_list)

# 3. Process Dataset
print("1/3: Cleaning ingredient text...")
df['Cleaned_Ingredients'] = df['Ingredients'].apply(clean_ingredients)
df = df.dropna(subset=['Cleaned_Ingredients'])
df = df.reset_index(drop=True)

# Save the cleaned data to be used by the website later
df.to_csv('../data/cleaned_recipes.csv', index=False)

# 4. Train ML Model (TF-IDF)
print("2/3: Training Machine Learning model...")
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['Cleaned_Ingredients'])

# 5. Save Model
print("3/3: Saving model files...")
joblib.dump(vectorizer, '../data/tfidf_vectorizer.pkl')
joblib.dump(tfidf_matrix, '../data/tfidf_matrix.pkl')
print("✅ DONE! Models are ready for the website.")

1/3: Cleaning ingredient text...
2/3: Training Machine Learning model...
3/3: Saving model files...
✅ DONE! Models are ready for the website.


In [14]:
# Save the cleaned dataset so we don't have to run the cleaner every time
df.to_csv('../data/cleaned_recipes.csv', index=False)

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib

# 0. Drop any empty rows just in case the cleaning process left blanks
df = df.dropna(subset=['Cleaned_Ingredients'])
df = df.reset_index(drop=True)

# 1. Initialize and fit the TF-IDF Vectorizer
print("Training TF-IDF Vectorizer...")
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['Cleaned_Ingredients'])
print(f"Matrix Shape: {tfidf_matrix.shape} (Rows: Recipes, Columns: Unique Ingredients)")

# 2. Build the Recommendation Function
import re

def recommend_recipes_smart_sort(user_ingredients_list, recipe_df, vectorizer_model, matrix, top_k=5):
    # 1. Clean the user inputs using the underscore trick we established
    cleaned_user_items = [clean_ingredients(item).strip() for item in user_ingredients_list]
    cleaned_user_items = [item for item in cleaned_user_items if item]
    user_string = " ".join(cleaned_user_items)
    
    # 2. Vectorize and calculate Cosine Similarity (The Soft Matcher)
    user_vector = vectorizer_model.transform([user_string])
    similarity_scores = cosine_similarity(user_vector, matrix).flatten()
    
    # 3. Grab a slightly larger pool of top matches (e.g., top 50) so we have enough to sort
    top_50_indices = similarity_scores.argsort()[-50:][::-1]
    top_results = recipe_df.iloc[top_50_indices].copy()
    top_results['Cosine_Score'] = similarity_scores[top_50_indices]
    
    # 4. The Counter: Count how many inputted ingredients actually appear in the recipe
    def count_matches(recipe_ingredients_string):
        count = 0
        for item in cleaned_user_items:
            # We use \b (word boundary) so it matches the exact token
            if re.search(rf'\b{item}\b', str(recipe_ingredients_string)):
                count += 1
        return count
        
    top_results['Match_Count'] = top_results['Cleaned_Ingredients'].apply(count_matches)
    
    # 5. The Sorter: Sort by Match_Count first (descending), then by Cosine_Score
    final_results = top_results.sort_values(
        by=['Match_Count', 'Cosine_Score'], 
        ascending=[False, False]
    ).head(top_k)
    
    return final_results[['Title', 'Ingredients', 'Match_Count', 'Cosine_Score']]

# --- Test the Smart Sorter ---
test_pantry = ['chicken', 'garlic', 'rice vinegar', 'soy sauce']
print(f"Searching for recipes prioritizing ingredient overlap: {test_pantry}\n")

results = recommend_recipes_smart_sort(test_pantry, df, vectorizer, tfidf_matrix, top_k=5)
display(results)

# 4. Save the models for your Web App (Streamlit)
joblib.dump(vectorizer, '../data/tfidf_vectorizer.pkl')
joblib.dump(tfidf_matrix, '../data/tfidf_matrix.pkl')
print("\n✅ Models successfully saved as .pkl files!")

Training TF-IDF Vectorizer...
Matrix Shape: (13501, 46085) (Rows: Recipes, Columns: Unique Ingredients)
Searching for recipes prioritizing ingredient overlap: ['chicken', 'garlic', 'rice vinegar', 'soy sauce']



,Title,Ingredients,Match_Count,Cosine_Score
119,Pajeon Sauce,"['2 tablespoons water', '2 tablespoons rice vi...",2,0.293310
4582,Spicy Roasted Cauliflower with Sriracha and Se...,"['1/3 cup olive oil', '1 teaspoon sesame oil',...",2,0.268800
7392,Candied Sweet Potato,['1/4 cup granulated or packed light brown sug...,2,0.268573
10411,Chinese Barbecued Baby Back Ribs,"['3 tablespoons chopped peeled ginger', '2 tab...",2,0.253639
4369,Sweet 'n' Spicy Sriracha-Glazed Salmon,['1/4 cup reduced-sodium soy sauce (or tamari ...,2,0.248239



✅ Models successfully saved as .pkl files!
